# ✋ Cliente 3 — Control por Gestos de Mano (webcam en Colab)

**Nivel: avanzado**

Este es el cliente más ambicioso del taller: usamos **MediaPipe Hands** para detectar tu mano con la cámara, y según la posición del dedo índice (arriba/abajo) mandamos comandos `+`/`-` al servidor — igual que el script original de escritorio con OpenCV, pero corriendo **dentro del navegador**, sin instalar nada.

### ⚠️ Cómo funciona (y su límite)
Colab no puede abrir tu cámara directamente como lo hace OpenCV en tu computadora (`cv2.VideoCapture`). En su lugar, usamos un puente de JavaScript: el navegador captura un cuadro de video, lo manda a Python, MediaPipe lo procesa, y el resultado se dibuja de vuelta sobre el video — cuadro por cuadro. Esto es más lento que una cámara local (unos pocos cuadros por segundo), pero es suficiente para mandar comandos de control.

**Requisitos:** usa Chrome o Edge, y acepta el permiso de cámara cuando el navegador lo pida.

In [ ]:
!pip install -q mediapipe opencv-python-headless socketIO-client

## 1. Conectarse al servidor

In [ ]:
from socketIO_client import SocketIO

SERVER_IP = "TU-IP-SERVIDOR"
SERVER_PORT = 5001
CONNECT_TO_SERVER = True

socketIO = None
if CONNECT_TO_SERVER:
    print("Conectando...")
    try:
        socketIO = SocketIO(SERVER_IP, SERVER_PORT)
        print("✅ Conectado al servidor.")
    except Exception as e:
        print(f"❌ Error al conectar: {e}")

## 2. Puente de cámara (JavaScript ↔ Python)

Esta celda define las funciones que le piden al navegador un cuadro de video y lo regresan a Python como imagen. No hace falta entenderla a fondo para usarla — solo ejecútala.

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode, b64encode
import io
import numpy as np
import PIL.Image
import cv2


def js_to_image(js_reply):
    image_bytes = b64decode(js_reply.split(",")[1])
    jpg_as_np = np.frombuffer(image_bytes, dtype=np.uint8)
    return cv2.imdecode(jpg_as_np, flags=1)


def overlay_to_bytes(overlay_array):
    overlay_pil = PIL.Image.fromarray(overlay_array, "RGBA")
    buf = io.BytesIO()
    overlay_pil.save(buf, format="png")
    return "data:image/png;base64," + str(b64encode(buf.getvalue()), "utf-8")


def video_stream():
    js = Javascript('''
        var video;
        var div = null;
        var stream;
        var captureCanvas;
        var imgElement;
        var labelElement;
        var pendingResolve = null;
        var shutdown = false;

        function removeDom() {
          stream.getVideoTracks()[0].stop();
          video.remove(); div.remove();
          video = null; div = null; stream = null;
          imgElement = null; captureCanvas = null; labelElement = null;
        }

        function onAnimationFrame() {
          if (!shutdown) { window.requestAnimationFrame(onAnimationFrame); }
          if (pendingResolve) {
            var result = "";
            if (!shutdown) {
              captureCanvas.getContext("2d").drawImage(video, 0, 0, 640, 480);
              result = captureCanvas.toDataURL("image/jpeg", 0.8);
            }
            var lp = pendingResolve;
            pendingResolve = null;
            lp(result);
          }
        }

        async function createDom() {
          if (div !== null) { return stream; }
          div = document.createElement("div");
          div.style.border = "2px solid black";
          div.style.padding = "3px";
          div.style.width = "100%";
          div.style.maxWidth = "640px";
          document.body.appendChild(div);

          const modelOut = document.createElement("div");
          modelOut.innerHTML = "<span>Estado:</span>";
          labelElement = document.createElement("span");
          labelElement.innerText = "Sin datos";
          labelElement.style.fontWeight = "bold";
          modelOut.appendChild(labelElement);
          div.appendChild(modelOut);

          video = document.createElement("video");
          video.style.display = "block";
          video.width = div.clientWidth - 6;
          video.setAttribute("playsinline", "");
          video.onclick = () => { shutdown = true; };
          stream = await navigator.mediaDevices.getUserMedia({video: true});
          div.appendChild(video);

          imgElement = document.createElement("img");
          imgElement.style.position = "absolute";
          imgElement.style.zIndex = 1;
          imgElement.onclick = () => { shutdown = true; };
          div.appendChild(imgElement);

          const instruction = document.createElement("div");
          instruction.innerHTML =
            "<span style=\\"color: red; font-weight: bold;\\">" +
            "Haz clic aquí o en el video para detener</span>";
          div.appendChild(instruction);
          instruction.onclick = () => { shutdown = true; };

          video.srcObject = stream;
          await video.play();

          captureCanvas = document.createElement("canvas");
          captureCanvas.width = 640;
          captureCanvas.height = 480;
          window.requestAnimationFrame(onAnimationFrame);
          return stream;
        }

        async function stream_frame(label, imgData) {
          if (shutdown) { removeDom(); shutdown = false; return ""; }
          await createDom();
          if (label != "") { labelElement.innerHTML = label; }
          if (imgData != "") {
            var videoRect = video.getClientRects()[0];
            imgElement.style.top = videoRect.top + "px";
            imgElement.style.left = videoRect.left + "px";
            imgElement.style.width = videoRect.width + "px";
            imgElement.style.height = videoRect.height + "px";
            imgElement.src = imgData;
          }
          var result = await new Promise(function(resolve) { pendingResolve = resolve; });
          shutdown = false;
          return result;
        }
        ''')
    display(js)


def video_frame(label, overlay):
    return eval_js(f'stream_frame("{label}", "{overlay}")')

## 3. Lógica de control por gestos

Misma idea que el script original: si la punta del dedo índice sube por encima de su nudillo (PIP), mandamos `+`; si baja, mandamos `-`. Cambia `MOTOR` para controlar otro motor.

In [ ]:
import mediapipe as mp
import time

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
hands = mp_hands.Hands(
    static_image_mode=False, max_num_hands=1, min_detection_confidence=0.5
)

MOTOR = "1"          # <-- cambia el motor a controlar ("1" a "5")
UMBRAL = 0.03          # sensibilidad del gesto (más chico = más sensible)
ENVIAR_CADA_S = 0.4    # para no saturar el servidor de mensajes

## 4. Ciclo principal

Ejecuta esta celda, acepta el permiso de cámara, y mueve tu mano frente a la cámara. Haz clic en el video (o en el texto rojo) para detener el ciclo.

In [ ]:
video_stream()
label_html = "Iniciando cámara..."
overlay = ""
ultimo_envio = 0

while True:
    js_reply = video_frame(label_html, overlay)
    if not js_reply:
        break

    img = js_to_image(js_reply)
    h, w = img.shape[:2]
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = hands.process(img_rgb)

    overlay_array = np.zeros([h, w, 4], dtype=np.uint8)
    comando = None

    if results.multi_hand_landmarks:
        hand_landmarks = results.multi_hand_landmarks[0]
        mp_drawing.draw_landmarks(overlay_array, hand_landmarks, mp_hands.HAND_CONNECTIONS)

        tip = hand_landmarks.landmark[mp_hands.HandLandmark.INDEX_FINGER_TIP]
        pip = hand_landmarks.landmark[mp_hands.HandLandmark.INDEX_FINGER_PIP]

        if tip.y < pip.y - UMBRAL:
            comando = "+"
        elif tip.y > pip.y + UMBRAL:
            comando = "-"

        ahora = time.time()
        if comando and socketIO and (ahora - ultimo_envio) > ENVIAR_CADA_S:
            socketIO.emit("ctrl_from_python", {"motor": MOTOR, "command": comando})
            ultimo_envio = ahora

    overlay_array[:, :, 3] = (overlay_array.max(axis=2) > 0).astype(np.uint8) * 255
    overlay = overlay_to_bytes(overlay_array)
    label_html = f"Motor {MOTOR} | Comando: {comando or '—'}"

## 🧪 Personaliza tu cliente

1. **Cambia de motor sin reiniciar**: usa un `input()` antes del ciclo, o crea un pequeño widget de Gradio/ipywidgets en paralelo para elegir el motor.
2. **Dos manos, dos motores**: detecta hasta 2 manos (`max_num_hands=2`) y controla un motor distinto con cada una, como en el script original.
3. **Velocidad variable**: si el dedo se mueve mucho respecto al cuadro anterior, manda pasos más grandes (gesto rápido = movimiento rápido).
4. **Otro gesto**: en vez de dedo arriba/abajo, usa la distancia entre pulgar e índice (pellizco) para controlar la posición.

### 🔧 Si algo no funciona
- **No pide permiso de cámara**: recarga la página y vuelve a correr la celda del ciclo principal.
- **Muy lento**: reduce la resolución en `captureCanvas.width/height` dentro del código JavaScript (por ejemplo, 480x360).
- **No detecta la mano**: acércate más a la cámara y busca buena iluminación — MediaPipe necesita ver bien los dedos.